In [1]:
import numpy as np
import glob
import os
import re
import sys

# This finds the project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

# --- THIS IS THE LINE YOU ARE MISSING ---
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
# --- ADD THAT LINE! ---

print(f"Project root added to path: {PROJECT_ROOT}")


Project root added to path: /Users/minglu/Documents/Uni/bistable_model_simulation


In [6]:
np.sqrt(2*1500*2*1e-7)

np.float64(0.02449489742783178)

In [26]:
from simulation.solvers.spatial_process import simul_initialize, simul_run
from simulation.solvers.rate_conversions import calculate_kappas


from pathlib import Path
import os

In [27]:
""" Main execution block containing all physics parameters. """
###### ================================== 1. parameter setting =====================================
L = 2. # cubic box length

diff_scale = 1500. 
DA = 1. 
DB = 1. 
DX = 1.  
DX2 = 1. 

##### There are 6 reactions but only 4 sigma values
##### because the reactions B <-> X involve no sigma value
sigmas = np.array((1., 1., 1., 1.)) * 0.1 # sigma_r1f, sigma_r1b, sigma_r2f, sigma_r2b

box_shape = np.array((L, L, L,))

##### the Part to change freely for the corresponding simulation
# Schloegl's model reaction rates
k = np.array((0.15, 0.025, 5.75, 25.))
print("Reaction rates for bistable schloegl's model: ",k)
# full model reaction rates
ls = np.array((1.5, 1500., 150., 25., 5.75, 25.))
print("Reaction rates for bistable full model: ",ls)
    

Reaction rates for bistable schloegl's model:  [ 0.15   0.025  5.75  25.   ]
Reaction rates for bistable full model:  [   1.5  1500.    150.     25.      5.75   25.  ]


In [28]:
range = [0.2, 0.5, 1.0, 2.0, 5.0, 10.0]

In [29]:
for i in range:
    print(f"Current diffusion coeff is: {diff_scale/i}.")
    diffusions = np.array((DX, DX2, DA, DB)) * diff_scale / i
    print(f"Diffusions are : {diffusions}.")
    kappas = calculate_kappas(ls, diffusions[2], diffusions[0], diffusions[1], sigmas)

Current diffusion coeff is: 7500.0.
Diffusions are : [7500. 7500. 7500. 7500.].
✅ R1 Solver converged.
✅ R2 Solver converged and passed physics checks.

--- Final Intrinsic Rates (Kappa) ---
κ₁⁺ = 7.1633e+02
κ₁⁻ = 1.5000e+03
κ₂⁺ = 3.6213e+04
κ₂⁻ = 6.0355e+03
κ₃⁺ = 5.7500e+00
κ₃⁻ = 2.5000e+01
The estimation is calculated based on the Eq. (32) in Erban's paper.
Current diffusion coeff is: 3000.0.
Diffusions are : [3000. 3000. 3000. 3000.].
✅ R1 Solver converged.
✅ R2 Solver converged and passed physics checks.

--- Final Intrinsic Rates (Kappa) ---
κ₁⁺ = 7.1654e+02
κ₁⁻ = 1.5000e+03
κ₂⁺ = 3.6835e+04
κ₂⁻ = 6.1392e+03
κ₃⁺ = 5.7500e+00
κ₃⁻ = 2.5000e+01
The estimation is calculated based on the Eq. (32) in Erban's paper.
Current diffusion coeff is: 1500.0.
Diffusions are : [1500. 1500. 1500. 1500.].
✅ R1 Solver converged.
✅ R2 Solver converged and passed physics checks.

--- Final Intrinsic Rates (Kappa) ---
κ₁⁺ = 7.1688e+02
κ₁⁻ = 1.5000e+03
κ₂⁺ = 3.7921e+04
κ₂⁻ = 6.3201e+03
κ₃⁺ = 5.7500e+00


In [30]:
diffusions = np.array((DX, DX2, DA, DB)) * 1500
kappas = calculate_kappas(ls, diffusions[2], diffusions[0], diffusions[1], sigmas)

✅ R1 Solver converged.
✅ R2 Solver converged and passed physics checks.

--- Final Intrinsic Rates (Kappa) ---
κ₁⁺ = 7.1688e+02
κ₁⁻ = 1.5000e+03
κ₂⁺ = 3.7921e+04
κ₂⁻ = 6.3201e+03
κ₃⁺ = 5.7500e+00
κ₃⁻ = 2.5000e+01
The estimation is calculated based on the Eq. (32) in Erban's paper.


In [31]:
def l1_plus_formula(kappa_1_plus, D, sigma):
     # Calculate the term inside the tanh function
    sqrt_term = np.sqrt(kappa_1_plus / (2 * D))
    
    # Calculate the Left-Hand Side (LHS) of the equation
    # This is the expression for the effective rate l_1^+
    tanh_val = np.tanh(sigma * sqrt_term)
    lhs = 4 * np.pi * D * (sigma - (1 / sqrt_term) * tanh_val)
    return lhs

def calculate_l2_rates(kappa_2_plus, kappa_2_minus, DA, DX, DX2, sigma_3):
    if kappa_2_plus <= 0 or kappa_2_minus <= 0: 
        return np.inf, np.inf
    
    alpha_sq = kappa_2_plus / (DX2 + DA) + kappa_2_minus / (DX2 + DX)
    alpha = np.sqrt(alpha_sq)
    common_factor = 4 * np.pi * (1 / alpha_sq) * (sigma_3 - np.tanh(alpha * sigma_3) / alpha)
    l2_plus = kappa_2_plus * common_factor
    l2_minus = kappa_2_minus * common_factor

    return l2_plus, l2_minus

In [ ]:
D = [20, 500, 750, 1000, 1500, 1600]
print(f"Kappas are {kappas}")
for k, _ in enumerate(D):
    print(f" ----- Current D is {D[k]} ----- ")
    print("Calculated l is:")
    l1p = l1_plus_formula(kappas[0], D[k], sigma=sigmas[0])
    l2p, l2m = calculate_l2_rates(kappas[2], kappas[3], D[k], D[k], D[k], sigmas[0])
    print(f"l1p:{l1p:.2f}, l2p:{l2p:.2f}, l2m:{l2m:.2f}")

Kappas are [7.16881808e+02 1.50000000e+03 3.79207338e+04 6.32012229e+03
 5.75000000e+00 2.50000000e+01]
 ----- Current D is 20 ----- 
Calculated l is:
l1p:1.40, l2p:30.16, l2m:5.03
 ----- Current D is 500 ----- 
Calculated l is:
l1p:1.50, l2p:135.00, l2m:22.50
 ----- Current D is 1000 ----- 
Calculated l is:
l1p:1.50, l2p:145.94, l2m:24.32
 ----- Current D is 1500 ----- 
Calculated l is:
l1p:1.50, l2p:150.00, l2m:25.00
 ----- Current D is 1600 ----- 
Calculated l is:
l1p:1.50, l2p:150.52, l2m:25.09
